In [ ]:
# ============================================================
# Download e unione dati ISTAT "Popolazione residente" (POSAS)
# Anni: 2020 - 2023 — link estratto via XPath dalla pagina
# ============================================================

!pip install lxml -q

import requests
from lxml import html
import zipfile
import io
import os
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


ANNI = [2020, 2021, 2022, 2023]
XPATH_LINK = '/html/body/div[1]/div/main/section[3]/div/div/table/tbody/tr[133]/td/a'
CARTELLA_DOWNLOAD = "/content/drive/MyDrive/Ricicla(MI)/ISTAT/istat_posas_download"
CARTELLA_ESTRAZIONE = "/content/drive/MyDrive/Ricicla(MI)/ISTAT/istat_posas_estratti"

os.makedirs(CARTELLA_DOWNLOAD, exist_ok=True)
os.makedirs(CARTELLA_ESTRAZIONE, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:

dataframes = []
errori = []

for anno in ANNI:
    print(f"\n--- Anno {anno} ---")
    page_url = f"https://demo.istat.it/app/?i=POS&a={anno}&l=it"

    zip_url = None

    # 1) Provo a leggere la pagina ed estrarre il link con lo XPath dato
    try:
        r_page = requests.get(page_url, headers=headers, timeout=30)
        r_page.raise_for_status()
        tree = html.fromstring(r_page.content)
        link_el = tree.xpath(XPATH_LINK)
        if link_el:
            href = link_el[0].get("href")
            zip_url = href if href.startswith("http") else f"https://demo.istat.it{href}"
            print(f"  Link trovato via XPath: {zip_url}")
    except Exception as e:
        print(f"  Attenzione: impossibile leggere la pagina ({e})")

    # 2) Fallback: se lo XPath non ha funzionato (es. pagina renderizzata via JS),
    #    costruisco l'URL secondo il pattern osservato
    if not zip_url:
        zip_url = f"https://demo.istat.it/data/posas/POSAS_{anno}_it_Tutti_i_file.zip"
        print(f"  Fallback su pattern URL: {zip_url}")

    # --- Download dello ZIP ---
    try:
        r = requests.get(zip_url, headers=headers, timeout=60)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  ERRORE nel download per l'anno {anno}: {e}")
        errori.append(anno)
        continue

    zip_path = os.path.join(CARTELLA_DOWNLOAD, f"POSAS_{anno}_Tutti_i_file.zip")
    with open(zip_path, "wb") as f:
        f.write(r.content)

    cartella_anno = os.path.join(CARTELLA_ESTRAZIONE, str(anno))
    os.makedirs(cartella_anno, exist_ok=True)

    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(cartella_anno)
            nomi_file = z.namelist()
    except zipfile.BadZipFile:
        print(f"  ERRORE: il file scaricato per l'anno {anno} non è uno ZIP valido.")
        errori.append(anno)
        continue

    csv_files = [
        f for f in nomi_file
        if f.lower().endswith(".csv")
        and "Ripartizioni" not in f
        and "Regioni" not in f
        and "Province" not in f
    ]

    print(f"  Estratti {len(nomi_file)} file, di cui {len(csv_files)} CSV. utili")

    for csv_name in csv_files:
        csv_path = os.path.join(cartella_anno, csv_name)
        try:
            df = pd.read_csv(
                csv_path,
                sep=";",
                # encoding="latin-1",
                skiprows=1,       # salta la prima riga (titolo descrittivo)
                header=0,         # la riga successiva (ora prima) è l'header vero
                low_memory=False
            )
        except Exception as e1:
            try:
                df = pd.read_csv(
                    csv_path,
                    sep=",",
                    encoding="utf-8",
                    skiprows=1,
                    header=0,
                    low_memory=False
                )
            except Exception as e2:
                print(f"  Impossibile leggere {csv_name}: {e1} / {e2}")
                continue
        # rinomino le colonne
        df.columns = ['codice_istat', 'comune', 'eta', 'celibi','coniugati','divorziati','vedovi', 'del_1', 'del_2','del_3', 'totale_maschi', 'nubili', 'coniugate','divorziate', 'vedove','del_4','del_5','del_6','totale_femmine','totale']

        df['anno_riferimento'] = anno
        # df["FILE_ORIGINE"] = csv_name

        # --- Rimuovo la riga di totale (Età = 999) ---
        col_eta = df.columns[2]
        df[col_eta] = pd.to_numeric(df[col_eta], errors="coerce")
        df = df[df[col_eta] != 999]
        # cancello le colonne vuote e marcate come "da eliminare"

        df = df.loc[:, ~df.columns.str.startswith('del_')]
        dataframes.append(df)

NameError: name 'ANNI' is not defined

In [ ]:
dataframes[0]

,codice_istat,comune,eta,celibi,coniugati,divorziati,vedovi,totale_maschi,nubili,coniugate,divorziate,vedove,totale_femmine,totale,anno_riferimento
0,1001,Agliè,0,5,0,0,0,5,11,0,0,0,11,16,2020
1,1001,Agliè,1,9,0,0,0,9,15,0,0,0,15,24,2020
2,1001,Agliè,2,6,0,0,0,6,8,0,0,0,8,14,2020
3,1001,Agliè,3,7,0,0,0,7,11,0,0,0,11,18,2020
4,1001,Agliè,4,13,0,0,0,13,15,0,0,0,15,28,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31818,1315,Volvera,96,0,0,0,1,1,0,0,0,5,5,6,2020
31819,1315,Volvera,97,0,0,0,1,1,1,0,0,2,3,4,2020
31820,1315,Volvera,98,0,0,0,0,0,0,0,0,3,3,3,2020
31821,1315,Volvera,99,0,0,0,0,0,0,0,0,0,0,0,2020


In [ ]:

# --- Unione finale ---
if dataframes:
    df_finale = pd.concat(dataframes, ignore_index=True)
    print(f"\n✅ Tabella finale creata: {df_finale.shape[0]} righe, {df_finale.shape[1]} colonne")
    display(df_finale.head())
    df_finale.to_csv("istat_popolazione_2020_2023.csv", index=False)
    print("File salvato come 'istat_popolazione_2020_2023.csv'")
else:
    print("\n⚠️ Nessun dato caricato.")

if errori:
    print(f"\n⚠️ Attenzione: problemi con gli anni: {errori}")


✅ Tabella finale creata: 6387644 righe, 15 colonne


,codice_istat,comune,eta,celibi,coniugati,divorziati,vedovi,totale_maschi,nubili,coniugate,divorziate,vedove,totale_femmine,totale,anno_riferimento
0,1001,Agliè,0,5,0,0,0,5,11,0,0,0,11,16,2020
1,1001,Agliè,1,9,0,0,0,9,15,0,0,0,15,24,2020
2,1001,Agliè,2,6,0,0,0,6,8,0,0,0,8,14,2020
3,1001,Agliè,3,7,0,0,0,7,11,0,0,0,11,18,2020
4,1001,Agliè,4,13,0,0,0,13,15,0,0,0,15,28,2020


File salvato come 'istat_popolazione_2020_2023.csv'
